In [ ]:
import numpy as np
import pandas as pd

import sys
myList = [[r'/PythonHelper/']]
myList.append(sys.path[1:])
myList = [aPath for aPart in myList for aPath in aPart]
sys.path = myList

import LASSI_2 as lassi
import BinTrjProc as BR
import TrjProc as TP

In [ ]:
# Parameters:

radial_bins = np.arange(0, 140.25, 0.25)
no_chains = 300
chain_length = 166
reps = [1,2,3]
box_L = 140

variants = ['SKGP', 'SSGP', 'SYGP']

# Compute radial density profiles computed for all residues, positively charged and negatively charged beads for each variant

In [1]:
def distance_ij(p1,p2,boxSize):

    halfBox=boxSize/2
    d=p2-p1
    d_x,d_y,d_z=[np.abs(ele) for ele in d]
    D_x=d_x if d_x<=halfBox else boxSize-d_x
    D_y=d_y if d_y<=halfBox else boxSize-d_y
    D_z=d_z if d_z<=halfBox else boxSize-d_z

    distance_wa=np.sqrt(D_x*D_x+D_y*D_y+D_z*D_z)
    
    return(distance_wa)

def center_of_mass(positions, box):
    N_p=len(positions) ###Find no. of atoms

    center_vector=[box[0][1]-box[0][0],box[1][1]-box[1][0],box[2][1]-box[2][0]]
    com=np.zeros(3)
    
    for dim in range(3):
        pos_dim=positions[:,dim]

        theta_dim=[(ele+center_vector[dim]/2)*np.pi*2/(center_vector[dim]) for ele in pos_dim]
        eta_dim=[np.cos(ele) for ele in theta_dim]
        zeta_dim=[np.sin(ele) for ele in theta_dim]
        mean_eta=np.mean(eta_dim)
        mean_zeta=np.mean(zeta_dim)

        mean_theta=np.arctan2(-mean_zeta,-mean_eta)+np.pi
        com[dim]=center_vector[dim]*mean_theta/2/np.pi
        com[dim]-=(center_vector[dim]/2)

        if (com[dim]<box[dim][0]):
            com[dim]+=center_vector[dim]

    return(com)

def flatten_2d_list(lst):
    return [item for sublist in lst for item in sublist]

In [ ]:

for s in range(0,len(variants)):
    print('Collecting data for: ', variants[s])

    tmp = np.arange(1,301,1)
    positive = []
    negative = []
    allbeads = []
    
    system_path = '/SimulationData/T55_'+variants[s]
    
    for r in range(0,len(reps)):
        trj_path = system_path[s]+'/'+str(reps[r])+"/A1LCD_trj.lassi"
        Extractor_obj=BR.TrjExtractor(trj_path) 
        
        pos_distance_reps = []
        neg_distance_reps = []
        all_distance_reps = []

        for frame in range(0,50):
             
            coord_list=Extractor_obj.extract_specific_frame(frame+50)

            # determine the com of the system 
            particles_in_the_cluster=[coord_list[(ele-1)*chain_length:ele*chain_length].tolist() for ele in tmp]
            flattened_coordinates = flatten_2d_list([coord for coord in particles_in_the_cluster])
            coordinates_array = np.array(flattened_coordinates)
            com = center_of_mass(coordinates_array, [[0,box_L],[0,box_L],[0,box_L]])

            pos_distance = []
            neg_distance = []
            all_distance = []

            # compute radial distance of all residues
            for chain in range(0,no_chains):
                for bead in range(0,chain_length):
                    all_distance.append(distance_ij(particles_in_the_cluster[chain][bead],com,box_L))
                
            if variants[s] == 'SKGP':
                Rs = [1,5,13,21,29,37,45,53,61,69,77,85,93,101,109,117,125,133,141,149,157]
                Ds = [7,15,23,31,39,47,55,63,71,79,87,95,103,111,119,127,135,143,151,159]
            else:
                Rs = [5,13,21,29,37,45,53,61,69,77,85,93,101,109,117,125,133,141,149,157]
                Ds = [7,15,23,31,39,47,55,63,71,79,87,95,103,111,119,127,135,143,151,159]
                
            for chain in range(0,no_chains):
                for pos in range(0,len(Rs)):
                    pos_distance.append(distance_ij(particles_in_the_cluster[chain][Rs[pos]],com,box_L))

            for chain in range(0,no_chains):
                for neg in range(0,len(Ds)):
                    neg_distance.append(distance_ij(particles_in_the_cluster[chain][Ds[neg]],com,box_L))
            
            All_hist,r=np.histogram(all_distance,bins=np.arange(0,140.25,0.25), density=False)
            Pos_hist,r=np.histogram(pos_distance,bins=np.arange(0,140.25,0.25), density=False)
            Neg_hist,r=np.histogram(neg_distance,bins=np.arange(0,140.25,0.25), density=False)

            all_distance_reps.append(All_hist)
            pos_distance_reps.append(Pos_hist)  
            neg_distance_reps.append(Neg_hist)  

        allbeads.append(np.mean(all_distance_reps, axis=0))
        positive.append(np.mean(pos_distance_reps, axis=0))     
        negative.append(np.mean(neg_distance_reps, axis=0))        
    
    all_array = np.mean(allbeads, axis=0)
    positive_array = np.mean(positive, axis=0)
    negative_array = np.mean(negative, axis=0)

#     np.save(save_path+'{c}_all_radial_count_V3.npy'.format(c=variants[s]),all_array)
#     np.save(save_path+'{c}_pos_radial_count_V3.npy'.format(c=variants[s]),positive_array)
#     np.save(save_path+'{c}_neg_radial_count_V3.npy'.format(c=variants[s]),negative_array)